In [0]:
#Data Loading
import pandas as pd
df = pd.read_csv("/Workspace/STDQ6324_Data_Management/FAOSTAT_data_new.csv")
df.head()

In [0]:
#Keep only relevant columns
df = df[['Area', 'Item', 'Element', 'Year', 'Value']]
df.head()

In [0]:
df.info()

In [0]:
#Check for duplication
df.duplicated().sum()

In [0]:
#Exploratory Data Analysis
df['Item'].unique()

In [0]:
df['Element'].unique()

In [0]:
#number of unique countries
df['Area'].nunique()

In [0]:
#Create food categories
def food_category(item):
    if item in ['Cereals - Excluding Beer',
                'Pulses']:
        return 'plant-based'
    elif item in ['Meat', 'Eggs', 'Fish, Seafood']:
        return 'animal-based'
    elif item == 'Sugar & Sweeteners':
        return 'sugar'
    else:
        return 'other'
df['Food_Type'] = df['Item'].apply(food_category)
df.head()

In [0]:
#Verification
df[['Item', 'Food_Type']].drop_duplicates()

In [0]:
#Apache Hive
#Convert Pandas DF into Spark DF
spark_df = spark.createDataFrame(df)
spark_df.createOrReplaceTempView("food")

In [0]:
%sql
SELECT *
FROM food
LIMIT 10

In [0]:
# Query 1: Sugar Supply Over Time
sugar_trend = spark.sql("""

SELECT Year, ROUND(AVG(Value),2) AS Avg_Sugar_Supply
FROM food
WHERE Element = 'Food supply (kcal/capita/day)'
AND Item = 'Sugar & Sweeteners'
GROUP BY Year
ORDER BY Year

""")
display(sugar_trend)

In [0]:
#Convert to Pandas
sugar_trend_df = sugar_trend.toPandas()

#Visualization
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
sns.lineplot(x='Year', y='Avg_Sugar_Supply', data=sugar_trend_df, color = "#4CC9F0", marker='o')
plt.title('Global Sugar Supply (2010-2023)')
plt.xlabel('Year')
plt.ylabel('Average Sugar Supply (kcal/capita/day)')
plt.show()

In [0]:
# Query 2: Top 10 Countries by Sugar Supply
sugar_supply = spark.sql("""

SELECT Area, ROUND(AVG(Value),2) AS Avg_Sugar_Supply
FROM food
WHERE Element = 'Food supply (kcal/capita/day)'
AND Item = 'Sugar & Sweeteners'
GROUP BY Area
ORDER BY Avg_Sugar_Supply DESC
LIMIT 10

""")
display(sugar_supply)


In [0]:
#Convert to Pandas
sugar_supply_df = sugar_supply.toPandas()

#Visualization
plt.figure(figsize=(10,5))
sns.barplot(x='Avg_Sugar_Supply', y='Area', data=sugar_supply_df, color="#4CC9F0")
plt.title('Top 10 Countries by Sugar Supply')
plt.xlabel('Average Sugar Supply (kcal/capita/day)')
plt.ylabel('Country')
plt.show()

In [0]:
# Query 3: Animal vs Plant Protein Supply Over Time
protein_trend = spark.sql("""

SELECT Year, Food_Type, ROUND(AVG(Value),2) AS Avg_Protein_Supply
FROM food
WHERE Element = 'Protein supply quantity (g/capita/day)'
AND Food_Type IN ('animal-based', 'plant-based')
GROUP BY Year, Food_Type
ORDER BY Year

""")
display(protein_trend)

In [0]:
#Convert to Pandas
protein_trend_df = protein_trend.toPandas()

#Indexing to view relative growth of protein
base_yr = 100
indexed_df = protein_trend_df.copy()
indexed_df['index_value'] = indexed_df.groupby('Food_Type')['Avg_Protein_Supply'].transform(lambda x: x / x.iloc[0] * base_yr)

#Visualization
fig, axes = plt.subplots(1,2, figsize=(15,5))

## Original Protein Trend Plot
#Filter data
plant_df = protein_trend_df[protein_trend_df['Food_Type'] == 'plant-based']
animal_df = protein_trend_df[protein_trend_df['Food_Type'] == 'animal-based']

axes[0].plot(plant_df['Year'], plant_df['Avg_Protein_Supply'], marker='o', linestyle='-', color="#7209B7", label='Plant Protein')
axes[0].plot(animal_df['Year'], animal_df['Avg_Protein_Supply'], marker='*', linestyle='--', color="#F72585", label='Animal Protein')
axes[0].set_title('Protein Supply Over Time (2010-2023)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Average Protein Supply (g/capita/day)')
axes[0].legend()

## Indexed Growth Plot
plant_index = indexed_df[indexed_df['Food_Type'] == 'plant-based']
animal_index = indexed_df[indexed_df['Food_Type'] == 'animal-based']

axes[1].plot(plant_index['Year'], plant_index['index_value'], marker='o', linestyle='-', color="#7209B7", label='Plant Protein')
axes[1].plot(animal_index['Year'], animal_index['index_value'], marker='*', linestyle='--', color="#F72585", label='Animal Protein')
axes[1].set_title('Relative Growth of Protein Supply (Base Year=100)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Index Value')
axes[1].legend()

plt.tight_layout()
plt.show()

In [0]:
# Query 4: Top 10 Countries by Animal Protein Supply
animal_trend = spark.sql("""

SELECT Area, ROUND(AVG(Value),2) AS Avg_Animal_Protein_Supply
FROM food
WHERE Element = 'Protein supply quantity (g/capita/day)'
AND Food_Type = 'animal-based'
GROUP BY Area
ORDER BY Avg_Animal_Protein_Supply DESC
LIMIT 10

""")
display(animal_trend)

In [0]:
# Query 5: Top 10 Countries by Plant Protein Supply
plant_trend = spark.sql("""

SELECT Area, ROUND(AVG(Value),2) AS Avg_Plant_Protein_Supply
FROM food
WHERE Element = 'Protein supply quantity (g/capita/day)'
AND Food_Type = 'plant-based'
GROUP BY Area
ORDER BY Avg_Plant_Protein_Supply DESC
LIMIT 10

""")
display(plant_trend)

In [0]:
#Convert to Pandas
animal_trend_df = animal_trend.toPandas()
plant_trend_df = plant_trend.toPandas()

#Visualization
fig, axes = plt.subplots(1, 2, figsize=(15,5))
#Animal Protein Supply
sns.barplot(x='Avg_Animal_Protein_Supply', y='Area', data=animal_trend_df, ax=axes[0], color="#F72585")
axes[0].set_title('Top 10 Countries by Animal Protein Supply')
axes[0].set_xlabel('Average Animal Protein Supply (g/capita/day)')
axes[0].set_ylabel('Country')
#Plant Protein Supply
sns.barplot(x='Avg_Plant_Protein_Supply', y='Area', data=plant_trend_df, ax=axes[1], color="#7209B7")
axes[1].set_title('Top 10 Countries by Plant Protein Supply')
axes[1].set_xlabel('Average Plant Protein Supply (g/capita/day)')
axes[1].set_ylabel('Country')

plt.tight_layout()
plt.show()

In [0]:
## Query 6: Distribution of Calorie Supply by Food Type
food_supply = spark.sql("""
        
SELECT Area, Food_Type, ROUND(AVG(Value),2) AS avg_calorie_supply
FROM food
WHERE Element = 'Food supply (kcal/capita/day)'
GROUP BY Area, Food_Type

""")
display(food_supply)

In [0]:
#Convert to Pandas
food_supply_df = food_supply.toPandas()

#Visualization
plt.figure(figsize=(10,5))
sns.boxplot(x='Food_Type', y='avg_calorie_supply', data=food_supply_df, palette={'plant-based': "#7209B7", 'animal-based': "#F72585", 'sugar': "#4CC9F0"})
sns.stripplot(x='Food_Type', y='avg_calorie_supply', data=food_supply_df, color='black', alpha=0.3, size=2, jitter=True)
plt.title('Distribution of Calorie Supply Across Food Type')
plt.xlabel('Food Type')
plt.ylabel('Calorie Supply (kcal/capita/day)')
plt.show()